# Acompanhamento 2 - Pipeline e modelos preliminares

Este notebook documenta o segundo checkpoint do projeto de previsao de precos de imoveis.

- O foco e demonstrar o `pipeline.py` funcionando.
- Os modelos ainda sao preliminares, mas ja seguem um fluxo reprodutivel.
- O treino consome somente os datasets limpos em `data/processed/`.
- As metricas sao comparadas com o baseline informado pelo professor.

## 1. Escopo do Acompanhamento 2

De acordo com o PDF do projeto, este checkpoint deve apresentar:

- estrutura do `pipeline.py` rodando;
- resultados preliminares dos primeiros modelos;
- fluxo de pre-processamento integrado ao modelo;
- capacidade de ler uma base nova e retornar predicoes sem erro.

O objetivo aqui nao e encerrar a competicao, mas mostrar que a base tecnica para a entrega final ja esta funcionando.

## 2. Organizacao dos datasets

Os arquivos ficam separados por etapa para evitar mistura entre base bruta, base limpa e resultados do modelo.

- `data/treino.csv`: dataset bruto de treino.
- `data/teste_publico.csv`: dataset bruto de teste publico.
- `data/processed/treino_limpo.csv`: dataset limpo usado para treinar e validar os modelos.
- `data/processed/teste_publico_limpo.csv`: dataset limpo usado para testar o `pipeline.py`.
- `acompanhamentos/acomp2/metricas_preliminares.csv`: resultados dos modelos preliminares.
- `acompanhamentos/acomp2/predicoes_teste_publico.csv`: predicoes geradas pelo melhor modelo preliminar.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed").exists() and (candidate / "acompanhamentos" / "acomp2").exists():
            return candidate
    raise FileNotFoundError("Nao foi possivel localizar a raiz do projeto.")

ROOT = find_project_root(Path.cwd().resolve())
ACOMP_DIR = ROOT / "acompanhamentos" / "acomp2"
PROCESSED_DIR = ROOT / "data" / "processed"
TRAIN_CLEAN_PATH = PROCESSED_DIR / "treino_limpo.csv"
PUBLIC_TEST_CLEAN_PATH = PROCESSED_DIR / "teste_publico_limpo.csv"
BASELINE_RMSLE = 0.17543

sys.path.insert(0, str(ACOMP_DIR))

## 3. Conferencia dos dados limpos

Antes do treino, verificamos se os arquivos limpos existem e se estao no formato esperado.

- `treino_limpo.csv` deve conter a coluna alvo `SalePrice`.
- `teste_publico_limpo.csv` nao deve conter `SalePrice`.
- Ambos devem estar em `data/processed/`, pois foram produzidos pela etapa de limpeza.
- A modelagem nao deve depender diretamente dos CSVs brutos.

In [2]:
for path in [TRAIN_CLEAN_PATH, PUBLIC_TEST_CLEAN_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Dataset limpo nao encontrado: {path}")

train_df = pd.read_csv(TRAIN_CLEAN_PATH)
public_test_df = pd.read_csv(PUBLIC_TEST_CLEAN_PATH)

resumo_datasets = pd.DataFrame({
    "dataset": ["treino_limpo.csv", "teste_publico_limpo.csv"],
    "local": [str(TRAIN_CLEAN_PATH.relative_to(ROOT)), str(PUBLIC_TEST_CLEAN_PATH.relative_to(ROOT))],
    "linhas": [len(train_df), len(public_test_df)],
    "colunas": [train_df.shape[1], public_test_df.shape[1]],
    "possui_saleprice": ["SalePrice" in train_df.columns, "SalePrice" in public_test_df.columns],
    "nulos_total": [int(train_df.isna().sum().sum()), int(public_test_df.isna().sum().sum())],
})
resumo_datasets

,dataset,local,linhas,colunas,possui_saleprice,nulos_total
0,treino_limpo.csv,data/processed/treino_limpo.csv,1168,81,True,0
1,teste_publico_limpo.csv,data/processed/teste_publico_limpo.csv,1459,80,False,0


## 4. Estrutura do pipeline de modelagem

O script `treino_modelos.py` concentra o fluxo de treino e selecao do modelo.

- A base e dividida em treino e validacao com `random_state=42`.
- Variaveis numericas recebem imputacao por mediana e escalonamento.
- Variaveis categoricas recebem imputacao com `Ausente` e One-Hot Encoding.
- O alvo `SalePrice` e transformado com `log1p` durante o treino.
- As predicoes voltam para dolares com `expm1`.
- O melhor modelo e salvo em `modelo_acomp2.joblib`.

## 5. Modelos preliminares testados

Foram escolhidos modelos iniciais com complexidade crescente.

- `Ridge`: baseline linear regularizado.
- `RandomForest`: modelo de ensemble baseado em arvores.
- `GradientBoosting`: ensemble sequencial, geralmente forte para dados tabulares.

As metricas avaliadas sao:

- `RMSLE`: metrica principal da competicao.
- `RMSE`: erro quadratico medio em dolares.
- `MAE`: erro absoluto medio em dolares.
- `R2`: proporcao da variancia explicada.
- `tempo_treino_s`: tempo de treino em segundos.

In [3]:
from treino_modelos import main as treinar_modelos

metricas = treinar_modelos()
metricas

Metricas preliminares:
          modelo   rmsle        rmse         mae      r2  tempo_treino_s
GradientBoosting 0.13082 24924.03543 15463.93071 0.90389         2.65982
           Ridge 0.13646 26517.00733 16348.81276 0.89122         0.05142
    RandomForest 0.15663 34761.09787 19626.07254 0.81306         0.74147

Modelo selecionado: GradientBoosting
Modelo salvo em: /home/arthur/dev/unifor/ciência de dados/acompanhamentos/acomp2/modelo_acomp2.joblib
Predicoes publicas salvas em: /home/arthur/dev/unifor/ciência de dados/acompanhamentos/acomp2/predicoes_teste_publico.csv


,modelo,rmsle,rmse,mae,r2,tempo_treino_s
0,GradientBoosting,0.130815,24924.035429,15463.930708,0.903895,2.659824
1,Ridge,0.136462,26517.007327,16348.812759,0.891217,0.051420
2,RandomForest,0.156631,34761.097868,19626.072542,0.813062,0.741469


## 6. Comparacao com o baseline do professor

O arquivo `docs/metricas_baseline.txt` informa RMSLE de `0.17543` para a regressao linear simples do professor.

- Valores menores de RMSLE sao melhores.
- Um modelo supera o baseline quando `rmsle < 0.17543`.
- Essa comparacao e preliminar, pois aqui usamos validacao local.

In [4]:
comparacao_baseline = metricas.copy()
comparacao_baseline["baseline_professor_rmsle"] = BASELINE_RMSLE
comparacao_baseline["supera_baseline"] = comparacao_baseline["rmsle"] < BASELINE_RMSLE
comparacao_baseline

,modelo,rmsle,rmse,mae,r2,tempo_treino_s,baseline_professor_rmsle,supera_baseline
0,GradientBoosting,0.130815,24924.035429,15463.930708,0.903895,2.659824,0.17543,True
1,Ridge,0.136462,26517.007327,16348.812759,0.891217,0.051420,0.17543,True
2,RandomForest,0.156631,34761.097868,19626.072542,0.813062,0.741469,0.17543,True


## 7. Validacao do `pipeline.py`

O `pipeline.py` deve estar pronto para receber um CSV de entrada e retornar apenas as predicoes.

Nesta validacao, conferimos que:

- a funcao `predict()` aceita o CSV limpo de teste publico;
- a saida e um `np.ndarray`;
- a quantidade de predicoes e igual a quantidade de linhas do dataset;
- todas as predicoes sao finitas;
- o retorno nao inclui coluna de ID nem coluna de texto.

In [5]:
from pipeline import predict

predicoes = predict(PUBLIC_TEST_CLEAN_PATH)
validacao_pipeline = {
    "tipo_saida": type(predicoes).__name__,
    "linhas_preditas": len(predicoes),
    "linhas_entrada": len(public_test_df),
    "somente_valores_finitos": bool(np.isfinite(predicoes).all()),
    "min_predicao": float(predicoes.min()),
    "max_predicao": float(predicoes.max()),
}
validacao_pipeline

{'tipo_saida': 'ndarray',
 'linhas_preditas': 1459,
 'linhas_entrada': 1459,
 'somente_valores_finitos': True,
 'min_predicao': 45700.81103620755,
 'max_predicao': 514390.8829870194}

## 8. Amostra das predicoes geradas

As predicoes do teste publico ficam salvas dentro da pasta do acompanhamento.

- O arquivo e apenas uma evidencia para apresentacao.
- O retorno exigido pelo avaliador continua sendo o array produzido por `pipeline.py`.
- A ordem das linhas segue a ordem de `data/processed/teste_publico_limpo.csv`.

In [6]:
pd.read_csv(ACOMP_DIR / "predicoes_teste_publico.csv").head()

,Id,SalePrice_predito
0,1461,127033.891617
1,1462,157756.492378
2,1463,176509.452121
3,1464,188250.652703
4,1465,194304.241613


## 9. Conclusao preliminar

- O fluxo de modelagem consome os datasets limpos em `data/processed/`.
- O melhor modelo preliminar foi salvo em `modelo_acomp2.joblib`.
- O `pipeline.py` foi validado com a base limpa de teste publico.
- A etapa seguinte natural e testar novos hiperparametros e modelos mais fortes para a entrega final.